In [ ]:
import sys
sys.path.insert(0, '../lib')

import os
import scvi
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import anndata as ad
import pynndescent
import numba
import seaborn as sns
import torch
import pytorch_lightning.loggers
import rapids_singlecell as rsc
import sc_utils
import gc

import common_data

# Initial clustering

### Load HVG object

In [ ]:
adata = sc.read_h5ad(common_data._sc_root / '07a_hvg1000_clustered.h5ad')

In [7]:
cells = adata.obs_names

In [8]:
del adata

### Load raw object

In [ ]:
raw_adata = sc.read_h5ad(common_data._sc_root / '01b_trimmed.h5ad')

In [4]:
adata.shape

(2491432, 1000)

In [5]:
raw_adata.shape

(3017191, 25141)

### Subset raw_adata to have same cells as in adata

In [11]:
gc.collect()

3041

In [12]:
raw_adata = raw_adata[raw_adata.obs.index.isin(cells)].copy()

In [13]:
raw_adata.shape

(2491432, 25141)

In [ ]:
adata = sc.read_h5ad(common_data._sc_root / '07a_hvg1000_clustered.h5ad')

### Add raw counts layer to adata

In [15]:
raw_adata_subset = raw_adata[:, adata.var_names]

In [16]:
adata.layers["counts"] = raw_adata_subset.X.copy()

### Normalize the data in .raw using log normalization

In [17]:
sc.pp.normalize_total(raw_adata, target_sum=1e4)

In [18]:
sc.pp.log1p(raw_adata)

### Add raw counts into HVG object

In [19]:
adata.raw = raw_adata

In [ ]:
adata.write_h5ad(common_data._sc_root / '07b_hvg1000_clustered+counts.h5ad')